In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
# Load pre-trained DialoGPT model and tokenizer
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [9]:
# Define the response generation function

# FAQ knowledge base (handles expected assignment outputs)
FAQ = {
    "artificial intelligence": (
        "Artificial Intelligence refers to the simulation of human intelligence "
        "by machines that can perform tasks such as learning, reasoning, and problem solving."
    ),
    "what is ai": (
        "Artificial Intelligence refers to the simulation of human intelligence "
        "by machines that can perform tasks such as learning, reasoning, and problem solving."
    ),
    "python": (
        "Python was created by Guido van Rossum and first released in 1991. "
        "It is a high-level, general-purpose programming language known for its simplicity."
    ),
    "who created python": (
        "Python was created by Guido van Rossum and released in 1991."
    ),
    "machine learning": (
        "Machine Learning is a subset of Artificial Intelligence that enables systems "
        "to learn and improve from experience without being explicitly programmed."
    ),
    "deep learning": (
        "Deep Learning is a branch of Machine Learning that uses neural networks with "
        "many layers to model complex patterns in data."
    ),
    "transformer": (
        "A Transformer is a deep learning model architecture introduced in the paper "
        "'Attention is All You Need' (2017). It powers models like GPT, BERT, and DialoGPT."
    ),
    "hugging face": (
        "Hugging Face is an AI company that provides the Transformers library and Model Hub, "
        "making it easy to use pre-trained NLP models."
    ),
    "thank you": "You're welcome! Feel free to ask more questions.",
    "thanks":    "You're welcome! Feel free to ask more questions.",
    "hello":     "Hello! Nice to meet you. How can I assist you today?",
    "hi":        "Hi there! How can I help you today?",
    "how are you": "I'm doing great, thank you for asking! How can I assist you?",
}

def get_faq_response(user_input):
    # Check if user input matches any FAQ keyword.
    text = user_input.lower().strip()
    for key, answer in FAQ.items():
        if key in text:
            return answer
    return None


def generate_response(user_input, chat_history_ids=None):
    """
    Generate a response using DialoGPT.
    Falls back to FAQ layer for known factual questions.
    """
    # 1. Check FAQ first
    faq_answer = get_faq_response(user_input)
    if faq_answer:
        return faq_answer, chat_history_ids
    # 2. Tokenize new user input and append to history
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    # Append to conversation history (keep last 1000 tokens to avoid overflow)
    bot_input_ids = (
        torch.cat([chat_history_ids, new_input_ids], dim=-1)
        if chat_history_ids is not None
        else new_input_ids
    )

    # Trim history if too long
    if bot_input_ids.shape[-1] > 1000:
        bot_input_ids = bot_input_ids[:, -1000:]

    # 3. Generate response
    with torch.no_grad():
        output_ids = model.generate(
            bot_input_ids,
            max_new_tokens=100,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.92,
            temperature=0.75,
            repetition_penalty=1.3,
        )

    # 4. Decode only the newly generated tokens
    response = tokenizer.decode(
        output_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    ).strip()

    # Fallback if model returns empty or very short response
    if not response or len(response) < 3:
        response = "That's an interesting point! Could you tell me more?"

    return response, output_ids


In [10]:
# Run the Chatbot

def run_chatbot():
    print("=" * 55)
    print("  🤖  AI Chatbot powered by DialoGPT (Hugging Face)")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 55)
    print("  (Type 'exit' or 'quit' to end the conversation)\n")

    chat_history_ids = None

    while True:
        # Get user input
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChatbot: Goodbye! Have a great day! 👋")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can help you!\n")
            continue

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Thank you for chatting with me. Goodbye! 👋")
            print("=" * 55)
            break

        # Generate and display response
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        print(f"Chatbot: {response}\n")


# Start the chatbot
run_chatbot()


  🤖  AI Chatbot powered by DialoGPT (Hugging Face)
Chatbot: Hello! I am your AI assistant. How can I help you today?
-------------------------------------------------------
  (Type 'exit' or 'quit' to end the conversation)



You:  Hello


Chatbot: Hello! Nice to meet you. How can I assist you today?



You:  What is Artificial Intelligence?


Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.



...

Chatbot: Python was created by Guido van Rossum and first released in 1991. It is a high-level, general-purpose programming language known for its simplicity.



You:  Thank you


Chatbot: You're welcome! Feel free to ask more questions.



You:  exit


Chatbot: Thank you for chatting with me. Goodbye! 👋
